# 🔧 Tools: Letting a Model Call Your Code

## Learning Objectives
In this notebook, you will learn:
1. **What a tool is** - a schema the model reads plus a function you execute
2. **The `@tool` decorator** - turn any Python function into a bindable tool
3. **`bind_tools`** - attach tools to a model so it can *request* calls
4. **The execution loop** - the three steps LangChain runs for you inside an agent, done by hand

## Prerequisites
- Completed `1-langchainintro.ipynb` and `2-modelintegration.ipynb`
- `pip install langchain langchain-openai python-dotenv`
- A `.env` file with `OPENAI_API_KEY`

---
## 💡 Part 1: What Is a Tool?

Models can request calls to code that fetches data from a database, searches the web, or runs
a computation. A tool is a pairing of two things:

1. **A schema** — the tool's name, description, and argument definitions (a JSON schema). This
   is all the model ever sees, and it is how the model decides *whether* and *how* to call.
2. **A function or coroutine** — the code that actually runs. The model never executes it;
   **your** program does.

### Key Concepts:
- **The model only requests**: it emits a structured `tool_call`, it does not run anything
- **Your description is the API docs**: a vague docstring produces a model that calls the tool
  at the wrong times — treat it as prompt engineering, not a comment

---
## 🔑 Part 2: Environment Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load credentials and initialize the model
# ============================================================================
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "❌ Set OPENAI_API_KEY in your .env file"

model = init_chat_model("gpt-4.1")

print(f"✅ Environment loaded — using {type(model).__name__}")

In [ ]:
# ============================================================================
# SANITY CHECK: A plain call, no tools involved yet
# ============================================================================
response = model.invoke("Why do parrots talk?")
print("🤖", response.content[:300], "...")

---
## 🏷️ Part 3: Defining a Tool with `@tool`

The `@tool` decorator wraps a function into a `BaseTool`, deriving the schema automatically:

- **Name** ← the function name (`get_weather`)
- **Description** ← the docstring — this is what the model reads to decide when to call
- **Argument schema** ← the type hints (`location: str`)

`bind_tools()` then attaches those schemas to the model. It returns a **new** model object; the
original `model` stays tool-free.

> **Note**: binding does not make the model call tools — it makes calling them *possible*.
> The model still chooses per request.

In [ ]:
# ============================================================================
# TOOL DEFINITION: A stub weather tool
# ============================================================================
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    # Stubbed for the demo — swap in a real weather API call.
    return f"It's sunny in {location}"


# bind_tools returns a NEW model; `model` itself is unchanged.
model_with_tools = model.bind_tools([get_weather])

print(f"🔧 Tool name:        {get_weather.name}")
print(f"📋 Tool description: {get_weather.description}")
print(f"📋 Tool args schema: {get_weather.args}")

### 3.1 🔍 Inspecting a Tool Call

Ask a question the tool can answer and the model responds with a **tool call** rather than
prose. Note what `print(response)` shows: `content=''`, with the payload in `.tool_calls`.
That empty content is expected — the model had nothing to say, only something to request.

In [ ]:
# ============================================================================
# TOOL CALLING: The model requests a call instead of answering
# ============================================================================
response = model_with_tools.invoke("What's the weather like in Boston?")

print(f"📄 content:   {response.content!r}   <- empty: the payload is in tool_calls")
print(f"🔧 tool_calls:")
for tool_call in response.tool_calls:
    print(f"     Tool: {tool_call['name']}")
    print(f"     Args: {tool_call['args']}")
    print(f"     ID:   {tool_call['id']}")

---
## 🔁 Part 4: The Tool Execution Loop

An agent is essentially this three-step loop, run until the model stops requesting tools. Doing
it by hand once makes clear exactly what `create_agent` automates:

1. **Model generates tool calls** — append its `AIMessage` to the history
2. **You execute the tools** — append a `ToolMessage` per call
3. **Model sees the results** — and produces the final natural-language answer

### Key Insight:
Each `ToolMessage` carries the `tool_call_id` of the request it answers. That is how the model
matches results to requests when it asked for several tools at once — which is why step 2 passes
the whole `tool_call` dict to `.invoke()` rather than just the args.

In [ ]:
# ============================================================================
# STEP 1: Model generates tool calls
# ============================================================================
messages = [{"role": "user", "content": "What's the weather in Boston?"}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)  # the tool-call message becomes part of the history

print(f"🔧 Model requested {len(ai_msg.tool_calls)} tool call(s)")

In [ ]:
# ============================================================================
# STEP 2: Execute the tools and collect their results
# ============================================================================
for tool_call in ai_msg.tool_calls:
    # Passing the whole tool_call dict (not just the args) makes .invoke()
    # return a ToolMessage already stamped with the matching tool_call_id.
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

    print(f"🔧 {tool_call['name']}({tool_call['args']}) -> {tool_result.content}")

In [ ]:
# ============================================================================
# STEP 3: Pass the results back for the final response
# ============================================================================
final_response = model_with_tools.invoke(messages)

print("🤖", final_response.text)
# e.g. "The current weather in Boston is sunny."

---
## 📝 Summary

In this notebook, we learned:

### 1. What a Tool Is
- **Schema + function**: the model reads the schema; your program runs the function
- **The docstring is the contract**: it is the only guidance the model has about when to call

### 2. Defining and Binding
- **`@tool`**: derives name from the function, description from the docstring, arg schema from
  the type hints
- **`bind_tools([...])`**: returns a *new* model that is allowed to request those tools

### 3. Reading a Tool Call
- **`content` is empty** on a tool-calling `AIMessage` — the payload lives in `.tool_calls`
- **Each call has `name`, `args`, and `id`**

### 4. The Execution Loop
- **Three steps**: model requests → you execute and append `ToolMessage`s → model answers
- **`tool_call_id` matching**: pass the whole `tool_call` dict to `.invoke()` so the
  `ToolMessage` is stamped correctly
- **This is what `create_agent` automates** — plus the loop, error handling, and middleware

### Next Steps
- **`4-messages.ipynb`** — the message types (`System`, `Human`, `AI`, `Tool`), metadata, and
  token usage